# 13 — Niezależne etykiety spekulacyjnych stanów BTC (BSADF / GSADF)
**Główne zadanie:** utworzyć referencyjne stany nadmiernego wzrostu cen
bez definiowania ich przez te same filtry LPPLS, które wykorzystuje hybryda.

Implementacja: prawostronny ADF(0) ze stałą, supremum po dopuszczalnych
początkach okna (BSADF na siatce), Monte Carlo pod hipotezą random walk.
To *aproksymacja* BSADF (krok startów/końców); pełna procedura PSY może
wymagać innych opóźnień, specyfikacji trendu oraz krytycznych wartości.

Poziom referencyjny `bubble_state_gsadf` jest **stanem dostępnym na koniec
dnia**, utrzymywanym do następnego przeliczenia. Nie jest dowodem istnienia
bańki fundamentalnej ani prognozą pęknięcia bańki.

In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from bubble_tools import load_prices, monte_carlo_bsadf_reference, UNION_EPISODES

ROOT=Path('.')
OUT=ROOT/'13_wyniki_bubble'
OUT.mkdir(exist_ok=True,parents=True)

# Dla powtarzalności użyj tego SAMEGO autentycznego pliku cen co przy LPPLS.
PRICE_CANDIDATES = [ROOT/'btc_usd_2015_2024.csv']
PRICE_FILE = next((p for p in PRICE_CANDIDATES if p.exists()),None)
if PRICE_FILE is None:
    raise FileNotFoundError('Brak cen BTC. Umieść prawdziwy btc_usd_2015_2024.csv lub zbiór z notatnika 07.')

# Ustal przed eksperymentem i zamroź; nie dobieraj po obejrzeniu wyniku OOS.
MIN_WINDOW = 120   # obserwacje (BTC: dni kalendarzowe)
END_STEP = 7       # częstość ponownej oceny
START_STEP = 14    # kroki siatki dla początków okien
MAX_WINDOW = None  # None = expanding BSADF; wartość liczbowa = truncated BSADF
MC_REPS = 199      # do pilota można ustawić 49; finalnie zwiększyć np. 499–999
CRITICAL_Q = 0.95
SEED = 42

prices=load_prices(PRICE_FILE)
prices=prices.loc[prices.date.between('2015-01-01','2024-12-31')].reset_index(drop=True)
print(f'Źródło: {PRICE_FILE}; zakres {prices.date.min().date()}–{prices.date.max().date()}; N={len(prices)}')
if len(prices)<MIN_WINDOW+100: raise ValueError('Za krótki szereg cen')
if prices.date.diff().dropna().ne(pd.Timedelta(days=1)).any():
    raise ValueError('Brakujące dni BTC. Nie uzupełniaj interpolacją bez osobnej procedury.')

## 1. Symulowana wartość krytyczna i sygnał as-of-date
Każda obserwacja BSADF używa tylko cen z przeszłości i bieżącej daty.
Losowe ścieżki do krytycznych wartości NIE używają cen z przyszłości.
`gsadf_evaluated_today=0` oznacza przepisanie ostatniej znanej decyzji.

In [ ]:
daily,settings=monte_carlo_bsadf_reference(
    prices,min_window=MIN_WINDOW,endpoint_step=END_STEP,
    start_step=START_STEP,max_window=MAX_WINDOW,
    reps=MC_REPS,quantile=CRITICAL_Q,seed=SEED)
assert (daily.loc[daily.reference_available_at.notna(),'reference_available_at'] <=
        daily.loc[daily.reference_available_at.notna(),'date']).all()
assert daily.date.is_unique
assert daily.bubble_state_gsadf.dropna().isin([0,1]).all()

(daily.to_csv(OUT/'13_bubble_reference_daily.csv',index=False))
(OUT/'13_config.json').write_text(json.dumps(settings,indent=2,ensure_ascii=False),encoding='utf-8')
print('Liczba ocen BSADF:',int(daily.gsadf_evaluated_today.sum()))
print('Dni z etykietą:',int(daily.bubble_state_gsadf.notna().sum()))
print('Dni pozytywne wg BSADF:',int((daily.bubble_state_gsadf==1).sum()))
display(daily.tail())

## 2. Zgodność z historyczną bazą UNION (analiza wyłącznie opisowa)
UNION może częściowo pochodzić z literatury LPPLS i nie stanowi
w pełni niezależnego standardu prawdy. Zgodność nie jest predykcją.

In [ ]:
valid=daily.bubble_state_gsadf.notna()
a=daily.loc[valid,'bubble_state_gsadf'].astype(int)
b=daily.loc[valid,'bubble_union'].astype(int)
tp=int(((a==1)&(b==1)).sum()); fp=int(((a==1)&(b==0)).sum())
fn=int(((a==0)&(b==1)).sum()); tn=int(((a==0)&(b==0)).sum())
comparison=pd.DataFrame([{
    'days_labeled':int(valid.sum()),'TP':tp,'FP':fp,'FN':fn,'TN':tn,
    'precision_vs_union':tp/(tp+fp) if tp+fp else np.nan,
    'recall_vs_union':tp/(tp+fn) if tp+fn else np.nan,
    'agreement':(tp+tn)/int(valid.sum())
}])
comparison.to_csv(OUT/'13_validation_vs_union.csv',index=False)
display(comparison)

## 3. Wykres ceny i obydwu definicji stanów bańkowych

In [ ]:
fig,ax=plt.subplots(figsize=(15,6))
ax.plot(daily.date,prices.price,lw=1.0,color='black',label='BTC-USD (Close)')
for i,(name,first,last) in enumerate(UNION_EPISODES):
    ax.axvspan(pd.Timestamp(first),pd.Timestamp(last),color='salmon',alpha=.25,
               label='UNION (literatura)' if i==0 else None)
bsadf_state=daily.bubble_state_gsadf.fillna(0).astype(int).to_numpy()
starts=np.where((bsadf_state==1)&(np.r_[0,bsadf_state[:-1]]==0))[0]
ends=np.where((bsadf_state==1)&(np.r_[bsadf_state[1:],0]==0))[0]
for i,(start,end) in enumerate(zip(starts,ends)):
    ax.axvspan(daily.date.iloc[start],daily.date.iloc[end]+pd.Timedelta(days=1),
               color='steelblue',alpha=.20,label='BSADF (referencja)' if i==0 else None)
ax.set(title='BTC: referencyjne stany BSADF vs epizody UNION',xlabel='Data',ylabel='BTC-USD [USD]')
ax.legend(loc='upper left'); ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'13_01_price_union_bsadf.png',dpi=170);plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(15,4))
ax.plot(daily.date,daily.bsadf_score_last,label='BSADF, ostatnia ocena',color='navy',lw=1)
ax.plot(daily.date,daily.bsadf_critical_last,label=f'Krytyczna wartość MC ({CRITICAL_Q:.0%})',color='crimson',ls='--')
ax.set(xlabel='Data',ylabel='t-statystyka',title='Prawostronny BSADF i zmienna w czasie wartość krytyczna')
ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'13_02_bsadf_critical.png',dpi=170);plt.show()

print('Zapisano:',OUT.resolve())